# Geometrical Mesh Construction

This notebook shows how to define an optical waveguide and finite difference mesh using a geometrical description (`GeometryModel` object), which is then converted into a finite difference mesh using the `layoutmesh()` function.  This approach separates the geometrical description of the waveguide from the computational parameters, and allows for a greater variety of arbitrary shapes and sizes.  

**Features:**
- The materials are described with symbolic labels (`"SiO2"`, `"SiN"`, `"air"`) rather than refractive-index arrays.
- The refractive index (or, for anisotropic materials, the three principal indices) of each material is then stored in a lookup table (dict)
- Substrate, cladding, and slab layers may be specified as semi-infinite regions
- Finite-size dielectric regions are specified with geometrical shapes (polygons, discs, rectangles)
- Computational parameters:  resolution (dx,dy) and size of computational domain are treated separately from the geometrical waveguide description. 
- Geometrical dimensions are not quantized by the finite difference grid.  `layoutmesh()` returns sub-cell-averaged permittivity tensor arrays.  
- `layoutmesh(...,yee=True)` returns (`epsxx`, `epsyy`, `epszz`) at their appropriate Yee staggered-grid locations, with shapes
  `(ny+1, nx)`, `(ny, nx+1)`, `(ny+1, nx+1)` respectively

In this example, we compute the fundamental TE mode of a silicon nitride ridge waveguide, with a sidewall angle of 75 degrees:
<pre>
                      --w--
                      _____
               air   /     \            ├ rh   |
              ______/       \______     |      | 
                       SiN                     ├ h
              _____________________            |
                      SiO2
 </pre>
 Note that `layoutmesh()` uses appropriate sub-cell averaging to model the sidewall that does not align to the cartesian grid.

In [ ]:
# Enable automatic reloading of modules (IPython only)
%load_ext autoreload
%autoreload 2

import modesolver as mode
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

In [ ]:
# Materials and refractive indices
n1 = 1.46       # SiO2 (substrate)
n2 = 2.00       # Si3N4 (waveguide, ridge)
n3 = 1.00       # Air (upper cladding)

materials = {
    "SiO2"  :   n1,
    "SiN"   :   n2,
    "air"   :   n3
}

# Waveguide dimensions:
h =  700        # silcon nitride core height (nm)
w =  900        # width of ridge, measured at top (nm)
rh = 400        # ridge height /etch depth (nm)
sidewall = 75   # sidewall angle (degrees)

# Computational window extent:
bottom = 1400    # lower cladding (nm)
top = 1000       # upper cladding (nm)
side = 2000      # Space on side (nm)

# Grid spacing (nm)
dx = 5          # horizontal grid (nm)
dy = dx         # vertical grid (nm)

## Geometry Definition

In [ ]:
# Infinite slab of SiN, with SiO2 below and air above:
layers = mode.LayerStack([mode.Layer(-np.inf,0,"SiO2"), # semiinfinite lower cladding
                          mode.Layer(0,h-rh,"SiN")],    # SiN core layer infinite slab, thickness (h-rh)
                          default_label="air")          # background air (upper cladding)
# Trapezoidal ridge of silicon nitride
core = mode.polygon([(-w/2,h),
                     (w/2,h),
                     (w/2 + rh/np.tan(np.radians(sidewall)),h-rh),
                     (-w/2 - rh/np.tan(np.radians(sidewall)),h-rh),
                     (-w/2,h)],
                     "SiN")
# Create geometry describing waveguide
model = mode.GeometryModel(layers,[core])

## Build the Permittivity Mesh

In [ ]:
x = np.arange(0, w/2 + side + dx/2, dx)
y = np.arange(-bottom, h + top + dy/2, dy)

x, y, xc, yc, nx, ny, epsxx, epsyy, epszz = mode.layoutmesh(
    model, materials, x=x, y=y, yee=True
)
edges = mode.geometry.boundary_segments(model,x[0],x[-1],y[0],y[-1])

print("Mesh Parameters:")
print(f"nx = {xc.size}, x in [{x[0]}, {x[-1]}] nm")
print(f"ny = {yc.size}, y in [{y[0]}, {y[-1]}] nm")
print(f"epsxx shape: {epsxx.shape}  (Ex / Hy locations)")
print(f"epsyy shape: {epsyy.shape}  (Ey / Hx locations)")
print(f"epszz shape: {epszz.shape}  (Ez corner locations)")

## Solve for the Fundamental TE Mode

In [ ]:
wavelength = 1550   # vacuum wavelength (nm)
nmodes = 1          # number of modes to calculate

neff, Ex, Ey, Ezj, Hx, Hy, Hzj = mode.wgmodes_yee(
    wavelength, n2, nmodes, dx, dy,
    epsxx=epsxx, epsyy=epsyy, epszz=epszz,
    boundary='000E', collocate=True
)

print(f"neff = {neff.real:.6f}")

# normalize fields to the peak amplitude of Ex
Exmax = np.max(abs(Ex))
Ex /= Exmax
Ey /= Exmax
Ezj /= Exmax

## Plot the Mode Field (Ex)

In [ ]:
# Plot Ex (TE Mode)
levels = np.arange(-45,0,5)
fig, ax = plt.subplots()

ax.imshow(abs(Ex),origin='lower',extent=[x[0],x[-1],y[0],y[-1]],cmap='jet')
ax.contour(xc,yc,20*np.log10(abs(Ex)),levels,colors='white',negative_linestyles='solid', linewidths=1)
ax.set_aspect('equal')
ax.axis([min(x), max(x), min(y), max(y)])
ax.set_xlabel('x (nm)')
ax.set_ylabel('y (nm)')
ax.set_title(f'Ex (TE Mode)\n$ n_{{eff}} = {neff:.6f} $')
ax.add_collection(LineCollection(edges, colors='black', linewidths=1))
    
plt.show()